---
# PDF to Markdown Processor

Converts research papers (PDFs) to clean, structured Markdown files.
- Preserves document structure (headings, sections, lists)
- Removes images but retains alt text/captions
- Optimizes for downstream RAG processing
- Handles academic paper formatting

In [ ]:
NUM_FILES = 3

In [ ]:
import os
import json
import re
import fitz  # PyMuPDF
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import argparse
from dataclasses import dataclass
from tqdm import tqdm

@dataclass
class ProcessingStats:
    """Track processing statistics"""
    total_files: int = 0
    successful: int = 0
    failed: int = 0
    total_pages: int = 0
    images_removed: int = 0
    captions_preserved: int = 0

class PDFToMarkdownProcessor:
    
    
    def __init__(self):
        
        self.header_patterns = [
            r'^[A-Z\s]{3,}$',  # All caps headers
            r'^\d+\.?\s+[A-Z][^.!?]*$',  # Numbered sections
            r'^[IVX]+\.?\s+[A-Z][^.!?]*$',  # Roman numeral sections
        ]
        
        self.section_headers = {
            'abstract', 'introduction', 'background', 'methods', 'methodology',
            'results', 'discussion', 'conclusion', 'conclusions', 'references',
            'acknowledgments', 'acknowledgements', 'funding', 'conflicts',
            'ethics', 'data availability', 'supplementary', 'appendix'
        }
        
        self.caption_patterns = [
            r'(Figure|Fig\.?)\s*\d+[.:]\s*(.+?)(?=\n\n|\n[A-Z]|\Z)',
            r'(Table)\s*\d+[.:]\s*(.+?)(?=\n\n|\n[A-Z]|\Z)',
            r'(Chart|Graph|Diagram)\s*\d+[.:]\s*(.+?)(?=\n\n|\n[A-Z]|\Z)',
        ]
        
        self.stats = ProcessingStats()

    def extract_text_blocks(self, page) -> List[Dict]:
        
        blocks = []
        
        try:
            page_dict = page.get_text("dict")
            
            for block in page_dict.get("blocks", []):
                if "lines" in block:  # Text block
                    block_lines = []
                    font_sizes = []
                    
                    for line in block["lines"]:
                        line_text = ""
                        for span in line.get("spans", []):
                            text = span.get("text", "")
                            if text:
                                # Clean unicode issues immediately
                                text = self.clean_unicode(text)
                                line_text += text
                                font_sizes.append(span.get("size", 12))
                        
                        if line_text.strip():
                            block_lines.append(line_text.strip())
                    
                    if block_lines:

                        block_text = '\n'.join(block_lines)
                        avg_font_size = sum(font_sizes) / len(font_sizes) if font_sizes else 12
                        
                        blocks.append({
                            "text": block_text,
                            "bbox": block.get("bbox", [0, 0, 0, 0]),
                            "font_size": avg_font_size,
                            "type": "text"
                        })
                        
        except Exception as e:
            print(f"    - Warning: Error extracting blocks: {e}")
            text = page.get_text()
            if text.strip():
                text = self.clean_unicode(text)
                blocks.append({
                    "text": text,
                    "bbox": [0, 0, 100, 100],
                    "font_size": 12,
                    "type": "text"
                })
        
        return blocks

    def clean_unicode(self, text: str) -> str:

        # Dictionary of problematic characters and their replacements
        unicode_replacements = {
            '\u00A0': ' ',      # Non-breaking space
            '\u2000': ' ',      # En quad
            '\u2001': ' ',      # Em quad
            '\u2002': ' ',      # En space
            '\u2003': ' ',      # Em space
            '\u2004': ' ',      # Three-per-em space
            '\u2005': ' ',      # Four-per-em space
            '\u2006': ' ',      # Six-per-em space
            '\u2007': ' ',      # Figure space
            '\u2008': ' ',      # Punctuation space
            '\u2009': ' ',      # Thin space
            '\u200A': ' ',      # Hair space
            '\u200B': '',       # Zero width space
            '\u200C': '',       # Zero width non-joiner
            '\u200D': '',       # Zero width joiner
            '\u2028': '\n',     # Line separator
            '\u2029': '\n\n',   # Paragraph separator
            '\u202F': ' ',      # Narrow no-break space
            '\u205F': ' ',      # Medium mathematical space
            '\u3000': ' ',      # Ideographic space
            '\uFEFF': '',       # Zero width no-break space (BOM)
            # Common ligatures
            '\uFB00': 'ff',
            '\uFB01': 'fi',
            '\uFB02': 'fl',
            '\uFB03': 'ffi',
            '\uFB04': 'ffl',
            # Smart quotes
            '\u2018': "'",      # Left single quotation mark
            '\u2019': "'",      # Right single quotation mark
            '\u201A': "'",      # Single low-9 quotation mark
            '\u201C': '"',      # Left double quotation mark
            '\u201D': '"',      # Right double quotation mark
            '\u201E': '"',      # Double low-9 quotation mark
            # Dashes
            '\u2010': '-',      # Hyphen
            '\u2011': '-',      # Non-breaking hyphen
            '\u2012': '-',      # Figure dash
            '\u2013': '-',      # En dash
            '\u2014': '--',     # Em dash
            '\u2015': '--',     # Horizontal bar
        }
        
        for char, replacement in unicode_replacements.items():
            text = text.replace(char, replacement)
        
        return text

    def detect_header_level(self, text: str, font_size: float, avg_font_size: float) -> Optional[int]:
        """Detect if text is a header and return its level"""
        text_clean = text.strip()
        
        size_ratio = font_size / avg_font_size if avg_font_size > 0 else 1
        
        if size_ratio > 1.3:
            return 1
        
        if size_ratio > 1.1:
            return 2
        
        text_lower = text_clean.lower()
        
        if any(section in text_lower for section in self.section_headers):
            return 2
        
        for pattern in self.header_patterns:
            if re.match(pattern, text_clean):
                return 3
        
        if (len(text_clean) < 100 and 
            not text_clean.endswith(('.', '!', '?', ':')) and
            len(text_clean.split()) <= 10 and
            '\n' not in text_clean):  
            return 3
        
        return None

    def extract_captions(self, text: str) -> Tuple[str, List[str]]:
        captions = []
        cleaned_text = text
        
        for pattern in self.caption_patterns:
            matches = re.finditer(pattern, text, re.IGNORECASE | re.DOTALL)
            for match in matches:
                caption_type = match.group(1)
                caption_text = match.group(2).strip()
                
                # Clean up caption text
                caption_text = re.sub(r'\s+', ' ', caption_text)
                caption_text = caption_text[:500]  # Limit length
                
                captions.append(f"**{caption_type} Caption:** {caption_text}")
                self.stats.captions_preserved += 1
                
                # Remove from main text
                cleaned_text = cleaned_text.replace(match.group(0), '')
        
        return cleaned_text, captions

    def clean_text(self, text: str) -> str:
        # Remove page numbers and obvious artifacts
        lines = text.split('\n')
        cleaned_lines = []
        
        for line in lines:
            line = line.strip()
            
            # Skip empty lines temporarily
            if not line:
                cleaned_lines.append('')
                continue
            
            # Skip likely page numbers (standalone numbers)
            if re.match(r'^\d+$', line):
                continue
            
            # Skip very short lines that are likely artifacts (but keep normal short lines)
            if len(line) < 3 and not re.match(r'^[a-zA-Z0-9\s\-\.]+$', line):
                continue
            
            # Skip lines with mostly special characters
            special_char_ratio = len(re.sub(r'[a-zA-Z0-9\s]', '', line)) / max(len(line), 1)
            if special_char_ratio > 0.7:
                continue
            
            cleaned_lines.append(line)
        
        text = '\n'.join(cleaned_lines)
        
        text = re.sub(r'[ \t]+', ' ', text)
        
        text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
        
        lines = text.split('\n')
        lines = [line.strip() for line in lines]
        text = '\n'.join(lines)
        
        return text

    def structure_to_markdown(self, blocks: List[Dict], captions: List[str]) -> str:

        markdown_lines = []
        
        font_sizes = [block["font_size"] for block in blocks if block["type"] == "text"]
        avg_font_size = sum(font_sizes) / len(font_sizes) if font_sizes else 12
        
        current_section = None
        in_references = False
        
        for i, block in enumerate(blocks):
            if block["type"] != "text":
                continue
                
            text = block["text"].strip()
            font_size = block["font_size"]
            
            if not text:
                continue
            
            text_lower = text.lower()
            
            if any(section in text_lower for section in ['abstract', 'introduction', 'methods', 'results', 'discussion', 'conclusion', 'references']):
                if 'references' in text_lower:
                    in_references = True
                
                if markdown_lines and markdown_lines[-1] != "":
                    markdown_lines.append("")
                markdown_lines.append(f"## {text}")
                markdown_lines.append("")
                current_section = text_lower
                continue
            
            if in_references:
                markdown_lines.append(text)
                continue
            
            header_level = self.detect_header_level(text, font_size, avg_font_size)
            
            if header_level and len(text) < 200:  
                adjusted_level = min(header_level + 2, 4)
                
                if markdown_lines and markdown_lines[-1] != "":
                    markdown_lines.append("")
                markdown_lines.append(f"{'#' * adjusted_level} {text}")
                markdown_lines.append("")
            else:
                sentences = self.split_into_sentences(text)
                
                current_paragraph = []
                for sentence in sentences:
                    current_paragraph.append(sentence)
                    
                    if (len(current_paragraph) >= 3 and len(' '.join(current_paragraph)) > 300) or \
                    len(' '.join(current_paragraph)) > 500:
                        
                        markdown_lines.append(' '.join(current_paragraph))
                        markdown_lines.append("")  
                        current_paragraph = []
                
                if current_paragraph:
                    markdown_lines.append(' '.join(current_paragraph))
                    markdown_lines.append("")
        
        if captions:
            markdown_lines.extend(["", "## Figures and Tables", ""])
            for caption in captions:
                markdown_lines.append(caption)
                markdown_lines.append("")
        
        return '\n'.join(markdown_lines)

    def split_into_sentences(self, text: str) -> List[str]:
        """Split text into sentences for better paragraph structure"""
        # Simple sentence splitting (you might want to use nltk for better results)
        import re
        
        # Split on sentence endings, but be careful with abbreviations
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
        
        cleaned_sentences = []
        for sentence in sentences:
            sentence = sentence.strip()
            if sentence and len(sentence) > 10:  # Avoid very short fragments
                cleaned_sentences.append(sentence)
        
        return cleaned_sentences

    def process_pdf(self, pdf_path: str, metadata_path: str = None) -> Dict:
        try:
            print(f"  Processing: {os.path.basename(pdf_path)}")
            
            metadata = {}
            if metadata_path and os.path.exists(metadata_path):
                with open(metadata_path, 'r', encoding='utf-8') as f:
                    metadata = json.load(f)
            
            doc = fitz.open(pdf_path)
            all_blocks = []
            all_captions = []
            
            total_pages = len(doc)
            print(f"    - Extracting text from {total_pages} pages...")
            
            for page_num in range(total_pages):
                page = doc[page_num]
                self.stats.total_pages += 1
                
                blocks = self.extract_text_blocks(page)
                all_blocks.extend(blocks)
                
                image_list = page.get_images()
                self.stats.images_removed += len(image_list)
            
            doc.close()
            
            full_text = '\n\n'.join([block["text"] for block in all_blocks if block["type"] == "text"])
            
            cleaned_text, captions = self.extract_captions(full_text)
            all_captions.extend(captions)
            
            cleaned_text = self.clean_text(cleaned_text)
            
            cleaned_blocks = []
            paragraphs = cleaned_text.split('\n\n')
            
            for i, paragraph in enumerate(paragraphs):
                paragraph = paragraph.strip()
                if paragraph:
                    cleaned_blocks.append({
                        "text": paragraph,
                        "bbox": [0, 0, 100, 100],
                        "font_size": 12,
                        "type": "text"
                    })
            
            markdown_content = self.structure_to_markdown(cleaned_blocks, all_captions)
            
            markdown_content = self.clean_text(markdown_content)
            
            result = {
                "success": True,
                "markdown": markdown_content,
                "metadata": metadata,
                "stats": {
                    "pages": total_pages,
                    "images_removed": self.stats.images_removed,
                    "captions_preserved": len(all_captions),
                    "text_length": len(markdown_content)
                }
            }
            
            print(f"    - Success: {len(markdown_content)} chars, {len(all_captions)} captions preserved")
            self.stats.successful += 1
            
            return result
            
        except Exception as e:
            print(f"    - Error processing {pdf_path}: {e}")
            self.stats.failed += 1
            return {
                "success": False,
                "error": str(e),
                "markdown": "",
                "metadata": metadata if 'metadata' in locals() else {}
            }

    def process_directory(self, input_dir: str, output_dir: str, max_files: int = None):
        """Process all PDFs in a directory"""
        input_path = Path(input_dir)
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        pdf_files = list(input_path.glob("*.pdf"))
        if max_files:
            pdf_files = pdf_files[:max_files]
        
        self.stats.total_files = len(pdf_files)
        
        print(f"Found {len(pdf_files)} PDF files to process")
        
        for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):

            metadata_file = pdf_file.with_name(f"{pdf_file.stem}_meta.json")
            metadata_path = metadata_file if metadata_file.exists() else None
            
            result = self.process_pdf(str(pdf_file), str(metadata_path) if metadata_path else None)
            
            if result["success"]:

                md_file = output_path / f"{pdf_file.stem}.md"
                with open(md_file, 'w', encoding='utf-8') as f:
                    f.write(result["markdown"])
                
                meta_file = output_path / f"{pdf_file.stem}_processed_meta.json"
                enhanced_metadata = result["metadata"].copy()
                enhanced_metadata.update({
                    "processing_stats": result["stats"],
                    "source_pdf": str(pdf_file),
                    "processed_markdown": str(md_file)
                })
                
                with open(meta_file, 'w', encoding='utf-8') as f:
                    json.dump(enhanced_metadata, f, indent=2, ensure_ascii=False)
        
        self.print_stats()

    def print_stats(self):

        print(f"\n{'='*50}")
        print("PROCESSING COMPLETE")
        print(f"{'='*50}")
        print(f"Total files: {self.stats.total_files}")
        print(f"Successful: {self.stats.successful}")
        print(f"Failed: {self.stats.failed}")
        print(f"Success rate: {(self.stats.successful/self.stats.total_files*100):.1f}%")
        print(f"Total pages processed: {self.stats.total_pages}")
        print(f"Images removed: {self.stats.images_removed}")
        print(f"Captions preserved: {self.stats.captions_preserved}")

def main():
    parser = argparse.ArgumentParser(description="Convert PDFs to clean Markdown for RAG processing")
    parser.add_argument("--input-dir", type=str, required=True, help="Directory containing PDF files")
    parser.add_argument("--output-dir", type=str, required=True, help="Directory to save Markdown files")
    parser.add_argument("--max-files", type=int, help="Maximum number of files to process (for testing)")
    
    args = parser.parse_args()
    
    processor = PDFToMarkdownProcessor()
    processor.process_directory(args.input_dir, args.output_dir, args.max_files)

if __name__ == "__main__":

    if __name__ == "__main__":

        input_directory = "D:/PsyWiz/raw_data"
        output_directory = "D:/PsyWiz/processed_md"
        
        processor = PDFToMarkdownProcessor()
        processor.process_directory(input_directory, output_directory, max_files=NUM_FILES)

Found 2 PDF files to process


Processing PDFs:   0%|          | 0/2 [00:00<?, ?it/s]

  Processing: article_2.pdf
    - Extracting text from 12 pages...


Processing PDFs:  50%|█████     | 1/2 [00:04<00:04,  4.23s/it]

    - Success: 48211 chars, 6 captions preserved
  Processing: article_3.pdf
    - Extracting text from 15 pages...


Processing PDFs: 100%|██████████| 2/2 [00:04<00:00,  2.24s/it]

    - Success: 79699 chars, 8 captions preserved

PROCESSING COMPLETE
Total files: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Total pages processed: 27
Images removed: 6
Captions preserved: 14
